# 6.2.1 Supervised Fine-Tuning (SFT)

Simulate SFT using logistic regression as a proxy: compare base model vs fine-tuned on few examples. Plot accuracy vs data size and training loss curves.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(0)

X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

base = LogisticRegression(max_iter=500, random_state=42).fit(X_tr, y_tr)
base_acc = accuracy_score(y_te, base.predict(X_te))
print(f'Base model accuracy: {base_acc:.3f}')

In [ ]:
few_shot_sizes = [10, 20, 50, 100, 200, 500]
ft_accs = []
for n in few_shot_sizes:
    idx = rng.choice(len(X_tr), n, replace=False)
    ft = LogisticRegression(max_iter=500, random_state=42).fit(X_tr[idx], y_tr[idx])
    ft_accs.append(accuracy_score(y_te, ft.predict(X_te)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(base_acc, color='steelblue', linestyle='--', lw=2, label='Base (full data)')
ax.plot(few_shot_sizes, ft_accs, 'o-', color='darkorange', lw=2, label='Fine-tuned')
ax.set(xlabel='Fine-tuning Samples', ylabel='Accuracy', title='SFT: Accuracy vs Data Size')
ax.set_xscale('log'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/sft_comparison.png')
print('Saved sft_comparison.png')
for n, a in zip(few_shot_sizes, ft_accs):
    print(f'  n={n:4d}: acc={a:.3f}')